In [1]:
import gmsh

gmsh.initialize()

gmsh.model.add("glacier_2d_single_crack")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Lz = 125.0

# -------------------------------------------------------------------------
# Notch dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size [m]
# Non-adaptive uniform mesh
# -------------------------------------------------------------------------

h = 2.5

gmsh.option.setNumber("Mesh.MeshSizeMin", h)
gmsh.option.setNumber("Mesh.MeshSizeMax", h)
gmsh.option.setNumber("Mesh.MeshSizeFactor", 1.0)

# -------------------------------------------------------------------------
# 2D glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    0.0,
    Lx,
    Lz,
)

# -------------------------------------------------------------------------
# Single notch
#
# Centered at x = Lx / 2
# Starts from the top surface
#
#       z = Lz
#  -----------   -----------
#             | |
#             | |  lz
#             |_|
#
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_z0 = Lz - lz

notch = gmsh.model.occ.addRectangle(
    notch_x0,
    notch_z0,
    0.0,
    lx,
    lz,
)

# -------------------------------------------------------------------------
# Subtract notch
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.cut(
    [(2, glacier)],
    [(2, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical surface
# -------------------------------------------------------------------------

surface_tags = [
    tag
    for dim, tag in domain
    if dim == 2
]

gmsh.model.addPhysicalGroup(
    2,
    surface_tags,
    1,
)

gmsh.model.setPhysicalName(
    2,
    1,
    "GLACIER",
)

# -------------------------------------------------------------------------
# Boundary physical group
# -------------------------------------------------------------------------

curves = gmsh.model.getEntities(1)

curve_tags = [
    tag
    for dim, tag in curves
]

gmsh.model.addPhysicalGroup(
    1,
    curve_tags,
    1,
)

gmsh.model.setPhysicalName(
    1,
    1,
    "BOUNDARY",
)

# -------------------------------------------------------------------------
# Mesh settings
# -------------------------------------------------------------------------

gmsh.option.setNumber(
    "Mesh.MeshSizeFromPoints",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromCurvature",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeExtendFromBoundary",
    0,
)

# -------------------------------------------------------------------------
# Generate 2D triangular mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(2)

# -------------------------------------------------------------------------
# Output
# -------------------------------------------------------------------------

gmsh.write("Lx500_1C_2D_NA.msh")

gmsh.finalize()

Info    : Increasing process stack size (8176 kB < 16 MB)
Info    : Meshing 1D...                                                                                                   
Info    : [  0%] Meshing curve 5 (Line)
Info    : [ 20%] Meshing curve 6 (Line)
Info    : [ 30%] Meshing curve 8 (Line)
Info    : [ 40%] Meshing curve 9 (Line)
Info    : [ 60%] Meshing curve 10 (Line)
Info    : [ 70%] Meshing curve 11 (Line)
Info    : [ 80%] Meshing curve 12 (Line)
Info    : [ 90%] Meshing curve 13 (Line)
Info    : Done meshing 1D (Wall 0.00137471s, CPU 0.000676s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.369184s, CPU 0.340701s)
Info    : 11901 nodes 23808 elements
Info    : Writing 'Lx500_1C_2D_NA.msh'...
Info    : Done writing 'Lx500_1C_2D_NA.msh'


In [3]:
import meshio

mesh = meshio.read("Lx500_1C_2D_NA.msh")

points = mesh.points
cells = mesh.cells_dict["triangle"]

meshio.write("glacier_2d_single_crack.xdmf",
    meshio.Mesh(
        points=points[:, :2],
        cells={"triangle": cells},
    ),
)